In [ ]:
#!pip install PyMuPDF pdfplumber chromadb sentence-transformers python-dotenv

In [7]:
import os
from pathlib import Path

import chromadb
from chromadb.config import Settings
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction as STEmbedding


BASE_DIR = Path.cwd()
VECTORDB_PATH = os.getenv("VECTORDB_PATH", str(BASE_DIR / "vectordb"))
COLLECTION_NAME = "uhcg_plans"

embedding_function = STEmbedding(
    model_name="BAAI/bge-m3"
)


def get_uhcg_collection():
    client = chromadb.PersistentClient(
        path=VECTORDB_PATH,
        settings=Settings(anonymized_telemetry=False),
    )

    collection = client.get_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_function,
    )

    return collection

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 26378.42it/s]


In [4]:
from typing import TypedDict, List, Any
from langgraph.graph import StateGraph, END


class CompareState(TypedDict):
    question: str
    insurer: str
    targets: List[str]
    retrieved: Any
    answer: str


def extract_targets_node(state: CompareState) -> CompareState:
    question = state["question"]

    # 임시
    if "medical" in question.lower() and "dental" in question.lower():
        state["targets"] = ["medical claim", "dental claim"]
    else:
        state["targets"] = ["general benefit", "claim procedure"]

    return state


def retrieve_compare_node(state: CompareState) -> CompareState:
    collection = get_uhcg_collection()

    retrieved = {}

    for target in state["targets"]:
        result = collection.query(
            query_texts=[target],
            n_results=3,
            where={"insurer": state["insurer"]}
        )

        retrieved[target] = {
            "documents": result["documents"][0],
            "metadatas": result["metadatas"][0],
        }

    state["retrieved"] = retrieved
    return state


def generate_answer_node(state: CompareState) -> CompareState:
    context = ""

    for target, data in state["retrieved"].items():
        docs = "\n\n".join(data["documents"])
        context += f"\n\n[{target}]\n{docs}"

    state["answer"] = f"""
아래 내용을 바탕으로 비교 답변을 만들면 됩니다.

사용자 질문:
{state["question"]}

검색 결과:
{context}

답변 형식:
- 표로 차이 정리
- 각 항목별 사용 상황 설명
- 문서에 없는 내용은 없다고 말하기
"""
    return state


graph = StateGraph(CompareState)

graph.add_node("extract_targets", extract_targets_node)
graph.add_node("retrieve_compare", retrieve_compare_node)
graph.add_node("generate_answer", generate_answer_node)

graph.set_entry_point("extract_targets")
graph.add_edge("extract_targets", "retrieve_compare")
graph.add_edge("retrieve_compare", "generate_answer")
graph.add_edge("generate_answer", END)

compare_graph = graph.compile()

In [ ]:
#!pip install langgraph

In [8]:
result = compare_graph.invoke({
    "question": "UHC에서 medical claim이랑 dental claim 차이가 뭐야?",
    "insurer": "uhcg",
    "targets": [],
    "retrieved": {},
    "answer": ""
})

print(result["answer"])

NotFoundError: Collection [uhcg_plans] does not exist